# vessel-watch: the high-resolution pipeline on real Sentinel-2 imagery

Launcher only; all logic lives in the `vesselwatch` package.

Runs the detector and classifiers trained on ShipRSImageNet (0.12 to 6 m per pixel) on seven Sentinel-2 scenes of the Finnish coast (10 m per pixel) in which every vessel has been boxed by the Finnish Environment Institute, and scores the published Finnish model on the same scenes as a reference. Nothing is trained.

**Before running: store your Copernicus login as Colab secrets.** Click the key icon in the left sidebar, add two secrets named `CDSE_USERNAME` (the email you registered with at dataspace.copernicus.eu) and `CDSE_PASSWORD`, and switch on "Notebook access" for both. They stay in your Google account and are not written to the notebook or to Drive.

A GPU makes this faster but the T4 is plenty. The first run downloads about 1 GB of imagery to Drive; later runs reuse it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Settings

In [ ]:
REPO_URL = 'https://github.com/bprtdaniel/vessel-watch.git'
BRANCH = 'main'

RUNS_DIR = '/content/drive/MyDrive/5.Projects/runs'                  # trained models are read from here, results written here
S2_DIR = '/content/drive/MyDrive/5.Projects/Data/sentinel2_finland'  # downloaded scenes and annotations

DETECTOR = f'{RUNS_DIR}/yolo11s_obb/weights/best.pt'

## Copernicus login

In [ ]:
import os
from google.colab import userdata

os.environ['CDSE_USERNAME'] = userdata.get('CDSE_USERNAME')
os.environ['CDSE_PASSWORD'] = userdata.get('CDSE_PASSWORD')
print('Login found for', os.environ['CDSE_USERNAME'].split('@')[0][:3] + '...')

## Code

In [ ]:
import subprocess

REPO_DIR = '/content/vessel-watch'
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)

# Colab already ships torch, torchvision, pandas, scikit-learn, matplotlib, pyyaml and opencv
%pip install -q --no-deps -e /content/vessel-watch
%pip install -q ultralytics
%cd /content/vessel-watch
!git log --oneline -1

os.environ['VESSELWATCH_RUNS'] = RUNS_DIR

## Run

For each of the seven scenes: download it if needed, cut it into 320 x 320 pixel chips, run both detectors over every chip, compare with the labelled boxes, and let the level 1 and level 3 classifiers name what is in the labelled boxes and in the detections.

The first detector listed is the pipeline's own; `--reference` adds the published Finnish model.

In [ ]:
!python -m vesselwatch.detect.finland --name step_a --data-dir "$S2_DIR" --detector high_res="$DETECTOR" --reference --drop-class Dock --classifier configs/crops/level1_resnet101.yaml="$RUNS_DIR/crops_level1_resnet101/best.pt" --classifier configs/crops/level3_resnet101.yaml="$RUNS_DIR/crops_level3_resnet101/best.pt"

## Results

The table above is also saved as `sentinel2_step_a.json` in `RUNS_DIR`, and example chips with the labelled vessels (green) and each detector's boxes are in `sentinel2_step_a_examples/`. The cell below shows a few of them.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

for path in sorted(Path(RUNS_DIR, 'sentinel2_step_a_examples').glob('*.png'))[:6]:
    print(path.name)
    display(Image(filename=str(path)))